In [1]:
import psycopg2
from datetime import datetime

conn = psycopg2.connect(
    host="localhost",
    database="kenexaihackathon",
    user="postgres",
    password="09092002"
)

cursor = conn.cursor()

In [6]:
# -----------------------------
# LOAD GOLD DATA
# -----------------------------

from datetime import datetime, timezone
import time
import pandas as pd


df = pd.read_sql_query(
    "SELECT * FROM gold.fact_incidents ORDER BY incident_key",
    conn
)

# -----------------------------
# SPLIT DATASET
# -----------------------------

historical = df.sample(frac=0.5, random_state=42)
live_stream = df.drop(historical.index)

print("Historical loaded:", len(historical))
print("Live incidents:", len(live_stream))

# -----------------------------
# LOAD HISTORICAL INCIDENTS
# -----------------------------

for _, row in historical.iterrows():

    cursor.execute("""
        UPDATE gold.fact_incidents
        SET incident_status = 'RESOLVED'
        WHERE incident_key = %s
    """, (row["incident_key"],))

conn.commit()

print("Historical incidents initialized")

# -----------------------------
# TICKET CREATION FUNCTION
# -----------------------------

def create_ticket(incident):

    cursor.execute("""
        SELECT 1 FROM gold.incident_tickets
        WHERE incident_id = %s
    """, (incident["incident_key"],))

    if cursor.fetchone():
        return
        severity_priority = {"LOW": 1, "MEDIUM": 2, "HIGH": 3}
        if severity_priority.get(incident["severity"], 0) < 2:
            return
    cursor.execute("""
        INSERT INTO gold.incident_tickets
        (incident_id, device_name, issue, status)
        VALUES (%s,%s,%s,'OPEN')
    """,
    (
        incident["incident_key"],
        incident["device_key"],
        incident["alert_type_key"]
    ))

    conn.commit()

    print("🎫 Ticket created for incident:", incident["incident_key"])


# -----------------------------
# INCIDENT MONITOR
# -----------------------------

def monitor_incident(incident):

    start_time = datetime.now(timezone.utc)

    while True:

        now = datetime.now(timezone.utc)

        duration = (now - start_time).total_seconds()

        if duration > 60:

            print("🚨 Escalation triggered")

            create_ticket(incident)

            break

        time.sleep(5)


# -----------------------------
# LIVE INCIDENT STREAM
# -----------------------------

print("Starting live stream...")

for _, incident in live_stream.iterrows():

    cursor.execute("""
        UPDATE gold.fact_incidents
        SET incident_status = 'OPEN'
        WHERE incident_key = %s
    """, (incident["incident_key"],))

    conn.commit()

    print("⚡ New incident:", incident["incident_key"])

    monitor_incident(incident)

    time.sleep(2)


cursor.close()
conn.close()

print("Live simulation finished.")


C:\Users\Palak\AppData\Local\Temp\ipykernel_17324\4135245873.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(


Historical loaded: 12789
Live incidents: 12789
Historical incidents initialized
Starting live stream...
⚡ New incident: 2


KeyboardInterrupt: 